In [10]:
# !pip install -U sentence_transformers outlines langchain-classic langchain-community langchain-chroma langchain-core langchain-text-splitters requests rank_bm25 langchain_huggingface

In [11]:
import pandas as pd
import os
import ast
from datasets import Dataset

from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    SentenceTransformerModelCardData,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.sentence_transformer.evaluation import InformationRetrievalEvaluator
import torch
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


In [12]:
def context_query_answers():
    base_data_path = '/content/drive/MyDrive/Colab Notebooks/RAG/Data'
    train_dataframe = pd.read_excel(os.path.join(base_data_path,'train.xlsx'))
    val_dataframe = pd.read_excel(os.path.join(base_data_path,'val.xlsx'))
    test_dataframe = pd.read_excel(os.path.join(base_data_path,'test.xlsx'))

    dataframe_list = [train_dataframe,val_dataframe,test_dataframe]
    query_list = []
    context_list = []
    answer_list = []

    for dataf in dataframe_list:
        for ind in range(0,len(dataf)):
            context_list.extend([dataf['context'][ind]])
            query_list.extend([dataf['question'][ind]])
            answer_list.extend([dataf['answers'][ind]])


    answer_list = [ast.literal_eval(i)[0] for i in answer_list]
    return context_list,query_list,answer_list


def sent_find(context,ans):

    ctx_list = [i for i in context.split("\n") if i]
    for ind , sen in enumerate(ctx_list):
        if ans in sen:
            if ind+1 < len(ctx_list):
                return ctx_list[ind]+ ' '+ ctx_list[ind+1]
            else:
                return ctx_list[ind]

    return None
    print("Not found")
def queries_anchor_creation(context_list,query_list,answer_list):
    anchor = []
    positive = []
    for ind , ans in enumerate(answer_list):
        sentence = sent_find(context_list[ind],ans)
        if sentence:
            anchor.append(query_list[ind])
            positive.append(sentence)
        else:
            print(ind)
            print("Not Fund")
    return anchor,positive


def custom_dataset_creation(threshold):
    context_list,query_list,answer_list = context_query_answers()
    anchor,positive = queries_anchor_creation(context_list,query_list,answer_list)

    train_data_list = []
    for ind , value in enumerate(anchor[:threshold]):
        mydict = {}
        mydict['anchor'] = value
        mydict['positive'] = positive[ind]
        train_data_list.append(mydict)
    train_dataset = Dataset.from_list(train_data_list)

    eval_data_list = []
    for ind , value in enumerate(anchor[threshold:]):
        mydict = {}
        mydict['anchor'] = value
        mydict['positive'] = positive[threshold+ind]
        eval_data_list.append(mydict)
    eval_dataset = Dataset.from_list(eval_data_list)

    return train_dataset,eval_dataset

def relevent_docs_creation(eval_dataset):
    queries = {}
    corpus = {}
    relevant_docs = {}

    for ind , mydict in enumerate(eval_dataset):
        queries['q'+str(ind+1)] = mydict['anchor']
        corpus['d'+str(ind+1)] = mydict['positive']
        relevant_docs['q'+str(ind+1)] = 'd'+str(ind+1)

    return queries,corpus,relevant_docs


### Hugging Face Dataset
dataset = load_dataset("sentence-transformers/all-nli", "pair")
train_dataset  = dataset["train"].select(range(1000))
eval_dataset   = dataset["dev"].select(range(100))
test_dataset   = dataset["test"].select(range(100))


In [13]:
threshold  = 4000
train_dataset,eval_dataset = custom_dataset_creation(threshold)

In [14]:
print(train_dataset)
print(eval_dataset)


Dataset({
    features: ['anchor', 'positive'],
    num_rows: 4000
})
Dataset({
    features: ['anchor', 'positive'],
    num_rows: 1000
})


In [15]:
train_dataset[101]

{'anchor': 'How many attacks is one of the suspects believed to have conducted?',
 'positive': 'One of the suspects is believed to have conducted more than 12 attacks since July. "These operations show the ability of Iraqi Security Forces to repeatedly capture criminals who undermine the security of Iraq," said Lt. Col. Neil Harper, a U.S. military spokesman.'}

In [16]:
queries,corpus,relevant_docs = relevent_docs_creation(eval_dataset)

In [17]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device = 'cuda')
loss = MultipleNegativesRankingLoss(model)



# 5. (Optional) Specify training arguments
args = SentenceTransformerTrainingArguments(
    # Required parameter:
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_ratio=0.1,
    fp16=True,  # Set to False if GPU can't handle FP16
    bf16=False,  # Set to True if GPU supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicates
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=100,
    save_total_limit=2,
    logging_steps=100,
    dataloader_pin_memory=False
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
# 6. (Optional) Create an evaluator & evaluate the base model
ir_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="checking",
)
results = ir_evaluator(model)
results

{'checking_cosine_accuracy@1': 0.198,
 'checking_cosine_accuracy@3': 0.452,
 'checking_cosine_accuracy@5': 0.579,
 'checking_cosine_accuracy@10': 0.716,
 'checking_cosine_precision@1': 0.198,
 'checking_cosine_precision@3': 0.15066666666666667,
 'checking_cosine_precision@5': 0.116,
 'checking_cosine_precision@10': 0.072,
 'checking_cosine_recall@1': 0.051,
 'checking_cosine_recall@3': 0.11675,
 'checking_cosine_recall@5': 0.14986666666666668,
 'checking_cosine_recall@10': 0.18611666666666668,
 'checking_cosine_ndcg@10': 0.17636964257485802,
 'checking_cosine_mrr@10': 0.3552821428571426,
 'checking_cosine_map@100': 0.09594910973587631}

In [19]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    evaluator=ir_evaluator,
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Checking Cosine Accuracy@1,Checking Cosine Accuracy@3,Checking Cosine Accuracy@5,Checking Cosine Accuracy@10,Checking Cosine Precision@1,Checking Cosine Precision@3,Checking Cosine Precision@5,Checking Cosine Precision@10,Checking Cosine Recall@1,Checking Cosine Recall@3,Checking Cosine Recall@5,Checking Cosine Recall@10,Checking Cosine Ndcg@10,Checking Cosine Mrr@10,Checking Cosine Map@100
100,0.170494,0.165121,0.215000,0.485000,0.620000,0.740000,0.215000,0.161667,0.124200,0.074500,0.055033,0.124950,0.160533,0.192450,0.185384,0.377507,0.101281
200,0.148696,0.167663,0.221000,0.489000,0.603000,0.723000,0.221000,0.163000,0.120800,0.073000,0.056783,0.126033,0.155950,0.188533,0.184620,0.379129,0.101855
300,0.257213,0.142720,0.221000,0.503000,0.627000,0.749000,0.221000,0.167667,0.125600,0.075400,0.057167,0.129867,0.161950,0.194533,0.189146,0.386323,0.103846
400,0.244542,0.151056,0.228000,0.493000,0.611000,0.733000,0.228000,0.164333,0.122400,0.073900,0.058450,0.127200,0.157867,0.190700,0.187604,0.386577,0.104464
500,0.191170,0.147380,0.224000,0.491000,0.629000,0.744000,0.224000,0.163667,0.126200,0.074900,0.057533,0.126617,0.162950,0.193450,0.189131,0.387858,0.104519
600,0.199445,0.143170,0.207000,0.473000,0.607000,0.749000,0.207000,0.157667,0.122000,0.075500,0.053250,0.122117,0.157783,0.195367,0.184999,0.372060,0.100742
700,0.223367,0.144631,0.208000,0.489000,0.622000,0.741000,0.208000,0.163000,0.124600,0.074700,0.053333,0.126283,0.160950,0.193033,0.185657,0.376728,0.101962
800,0.164724,0.147132,0.214000,0.485000,0.627000,0.745000,0.214000,0.161667,0.125600,0.075200,0.054917,0.125083,0.162283,0.194367,0.186855,0.379200,0.102510
900,0.158330,0.156299,0.221000,0.478000,0.628000,0.737000,0.221000,0.159333,0.125800,0.074400,0.056750,0.123333,0.162450,0.191867,0.186175,0.379842,0.102421
1000,0.134339,0.148610,0.220000,0.486000,0.619000,0.742000,0.220000,0.162000,0.124000,0.074900,0.056417,0.125533,0.160033,0.193117,0.187038,0.381246,0.102801


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.18923207855224608, metrics={'train_runtime': 128.2094, 'train_samples_per_second': 31.199, 'train_steps_per_second': 7.8, 'total_flos': 0.0, 'train_loss': 0.18923207855224608, 'epoch': 1.0})

In [20]:
ir_evaluator(model)

{'checking_cosine_accuracy@1': 0.22,
 'checking_cosine_accuracy@3': 0.486,
 'checking_cosine_accuracy@5': 0.619,
 'checking_cosine_accuracy@10': 0.742,
 'checking_cosine_precision@1': 0.22,
 'checking_cosine_precision@3': 0.162,
 'checking_cosine_precision@5': 0.124,
 'checking_cosine_precision@10': 0.07490000000000001,
 'checking_cosine_recall@1': 0.056416666666666664,
 'checking_cosine_recall@3': 0.12553333333333333,
 'checking_cosine_recall@5': 0.16003333333333336,
 'checking_cosine_recall@10': 0.1931166666666667,
 'checking_cosine_ndcg@10': 0.18703820274682978,
 'checking_cosine_mrr@10': 0.3812456349206343,
 'checking_cosine_map@100': 0.10280116906619055}